# Notebook 10: Where INT8 fails and selective quantisation (E14, H4), abstention rules C1/C2 (H5), and the Phase 2 freeze

**Question (H4).** Full-integer INT8 cost the Phase 1 cassava model 9.6 points. Which layers are responsible, and does keeping the k worst layers in float give an integer model inside the Phase 1 budgets (≤10 MiB per crop, ≤20 MiB both; ≤2 pp accuracy and ≤0.02 macro-F1 lost versus FP32 on the selection partition)?

**Method.** The LiteRT quantisation debugger (Google AI Edge, n.d.-b) runs the float and the quantised model on the same representative images (the Phase 1 representative set) and reports, per operation, the quantisation error relative to its scale (`rmse/scale`). Operations are ranked by that error; the k worst are kept in float for k ∈ {0, 5, 10, 20, 40} ("selective INT8"). Every file is scored on the selection, temperature and threshold partitions only. MobileNetV2 (shift-augmented, E11b) full-INT8 is the comparator.

**Decision rules (registered, Section 5).** (1) v3 model per crop: the shift-augmented EfficientNet-B0 (E11b, seed 42) if its selection macro-F1 is within 0.02 of the Phase 1 model, else the Phase 1 model. (2) v3 format: the smallest file passing every budget among selective-INT8 (smallest passing k) and FP16. (3) v3 adaptation: as registered by Notebook 09. (4) v3 abstention: per-class thresholds (C1) if the target is met for every class, else the Phase 1 global rule; the energy score (C2, Liu et al., 2020) with the 95th-percentile rule is reported as a score only.

**Freeze.** `records/frozen_choices_v3.json` (+ SHA-256) is written at the end. Its existence unseals `d5_test` for Notebook 11. No test-half, challenge or D5 image is loaded here.

**Runtime.** Any GPU runtime (T4 is fine; the conversions and scoring run on the CPU). About 60–90 minutes. Resumable. Needs Notebook 09 finished (E11b models and `v3_adaptation_registered.json`).

In [ ]:
# 1. Setup, integrity checks, environment
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, glob, tarfile, hashlib, datetime, platform, subprocess, gc, shutil, io
import numpy as np, pandas as pd
import tensorflow as tf, keras
from sklearn.metrics import f1_score, accuracy_score
DRIVE = '/content/drive/MyDrive/ocdf'; WORK = '/content/working'
RUNS1 = f'{DRIVE}/runs'; RUNS = f'{DRIVE}/runs_v3'; OUT = f'{DRIVE}/records/nb10'; MODELS = f'{DRIVE}/models_v3'; LOGITS = f'{OUT}/logits'
for d in [OUT, MODELS, LOGITS, WORK]: os.makedirs(d, exist_ok=True)
txt_path = f'{DRIVE}/records/protocol_v3_registration_text.md'
assert hashlib.sha256(open(txt_path, 'rb').read()).hexdigest() == open(txt_path + '.sha256').read().strip(), 'registration text changed'
reg_sha = open(f'{DRIVE}/records/protocol_v3_registration.json.sha256').read().strip()
fpath = f'{DRIVE}/records/frozen_choices_v2.json'
assert hashlib.sha256(open(fpath, 'rb').read()).hexdigest() == open(f'{DRIVE}/records/frozen_choices_v2.sha256').read().strip(), 'frozen choices v2 changed'
F = json.load(open(fpath)); DEP = F['deployed_instance']; protocol2 = json.load(open(f'{DRIVE}/records/protocol_v2.json'))
adec_path = f'{DRIVE}/records/v3_adaptation_registered.json'
assert os.path.exists(adec_path), 'Run Notebook 09 cell 7 first (v3_adaptation_registered.json is missing)'
assert hashlib.sha256(open(adec_path, 'rb').read()).hexdigest() == open(adec_path + '.sha256').read().strip(), 'adaptation registration changed'
ADEC = json.load(open(adec_path)); print('v3 adaptation registered:', ADEC['maize'])
assert not os.path.exists(f'{DRIVE}/records/frozen_choices_v3.json'), 'frozen_choices_v3.json already exists: Phase 2 is frozen; do not rerun this notebook'
try:
    import ai_edge_litert; from ai_edge_litert.interpreter import Interpreter, OpResolverType; litert_pkg = f'ai_edge_litert {getattr(ai_edge_litert, "__version__", "")}'
except Exception:
    Interpreter = tf.lite.Interpreter; OpResolverType = tf.lite.experimental.OpResolverType; litert_pkg = 'tf.lite (bundled with TensorFlow)'
try: gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
except Exception: gpu_name = 'none'
env = {'notebook': '10_quantisation_freeze', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python': sys.version.split()[0],
       'tensorflow': tf.__version__, 'keras': keras.__version__, 'numpy': np.__version__, 'litert_interpreter': litert_pkg, 'gpu': gpu_name, 'cpu_count': os.cpu_count(), 'platform': platform.platform()}
json.dump(env, open(f'{OUT}/environment_nb10.json', 'w'), indent=1); print(json.dumps(env, indent=1))

In [ ]:
# 2. Manifest v3, development partitions only, models, decision rule 1 (v3 model per crop)
seal = json.load(open(f'{DRIVE}/records/phase2_data_v3.json'))
mpath = f'{DRIVE}/manifests/{seal["manifest_v3"]}'
assert hashlib.sha256(open(mpath, 'rb').read()).hexdigest() == seal['sha256'], 'manifest_v3 changed since it was frozen'
M = pd.read_csv(mpath, dtype=str)
ALLOWED = ['train', 'select', 'calib_T', 'thresh']                    # never test halves, challenge sets, D3 external or D5 here
U = M[M.partition.isin(ALLOWED)].copy(); del M
for src in ['D1', 'D4', 'D6']:
    if not os.path.isdir(f'{WORK}/{src}'):
        with tarfile.open(f'{DRIVE}/working/working_{src}.tar') as tar: tar.extractall(WORK, filter='data')
CLASSES = {'cassava': ['C01', 'C02', 'C03'], 'maize': ['M01', 'M02', 'M03', 'M04', 'M05']}; IMG = 224; KS = [0, 5, 10, 20, 40]
def load_img(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize_with_pad(img, IMG, IMG, method='bilinear')
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)
def load_array(df):
    ds = tf.data.Dataset.from_tensor_slices([f'{WORK}/{p}' for p in df.working_path]).map(load_img, num_parallel_calls=tf.data.AUTOTUNE).batch(256)
    return np.concatenate([b.numpy() for b in ds])
PARTS = {crop: {p: U[(U.crop == crop) & (U.partition == p)].sort_values('sample_id') for p in ['select', 'calib_T', 'thresh']} for crop in CLASSES}
# candidate models: Phase 1 EffB0 s42, E11b shift EffB0 s42, E11b shift MobileNetV2 s42 (comparator)
CAND = {}
for crop in CLASSES:
    CAND[(crop, 'phase1_EfficientNetB0')] = f'{RUNS1}/{crop}_EfficientNetB0_s42/best_model.keras'
    CAND[(crop, 'shift_EfficientNetB0')] = f'{RUNS}/E11b_{crop}_EfficientNetB0_shift_s42/best_model.keras'
    CAND[(crop, 'shift_MobileNetV2')] = f'{RUNS}/E11b_{crop}_MobileNetV2_shift_s42/best_model.keras'
    for k, p in CAND.items(): assert os.path.exists(p), f'missing {p} (run Notebook 09 first)'
E11B = {crop: {a: json.load(open(f'{RUNS}/E11b_{crop}_{a}_shift_s42/done.json')) for a in ['EfficientNetB0', 'MobileNetV2']} for crop in CLASSES}
# Decision rule 1 on Keras selection macro-F1 (recomputed here for both models with the same code)
SEL = {}; RULE1 = {}
for crop in CLASSES:
    se = PARTS[crop]['select']; y = np.array([CLASSES[crop].index(l) for l in se.research_label]); Xs = load_array(se)
    for tag in ['phase1_EfficientNetB0', 'shift_EfficientNetB0', 'shift_MobileNetV2']:
        keras.backend.clear_session(); m = keras.models.load_model(CAND[(crop, tag)])
        p = m.predict(Xs.astype(np.float32), batch_size=64, verbose=0)
        SEL[(crop, tag)] = {'select_macro_f1': float(f1_score(y, p.argmax(1), average='macro')), 'select_accuracy': float(accuracy_score(y, p.argmax(1)))}
        del m
    d = SEL[(crop, 'shift_EfficientNetB0')]['select_macro_f1'] - SEL[(crop, 'phase1_EfficientNetB0')]['select_macro_f1']
    RULE1[crop] = {'phase1_select_macro_f1': SEL[(crop, 'phase1_EfficientNetB0')]['select_macro_f1'], 'shift_select_macro_f1': SEL[(crop, 'shift_EfficientNetB0')]['select_macro_f1'],
                   'difference': d, 'v3_model': 'shift_EfficientNetB0' if d >= -0.02 else 'phase1_EfficientNetB0',
                   'rule': 'shift-augmented EfficientNet-B0 (seed 42) if its selection macro-F1 is within 0.02 of the Phase 1 model, else the Phase 1 model'}
    print(crop, 'rule 1:', json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in RULE1[crop].items() if k != 'rule'}))
    del Xs; gc.collect()
json.dump({'selection': {f'{c}/{t}': v for (c, t), v in SEL.items()}, 'rule1': RULE1}, open(f'{OUT}/rule1_model_choice.json', 'w'), indent=1)

In [ ]:
# ---- OCDF F4 calibration, abstention and selection helpers (Notebook 04) ----
import numpy as np

def softmax(z, T=1.0):
    z = np.asarray(z, dtype=np.float64) / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def nll(z, y, T=1.0):
    p = softmax(z, T)
    return float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1.0))))

def fit_temperature(z, y, lo=0.05, hi=20.0):
    """Calibration: positive T minimising negative log-likelihood on the calibration partition (Guo et al., 2017).
    Golden-section search on log T (NLL is unimodal in T for fixed logits)."""
    a, b = np.log(lo), np.log(hi); g = (np.sqrt(5) - 1) / 2
    c, d = b - g * (b - a), a + g * (b - a)
    fc, fd = nll(z, y, np.exp(c)), nll(z, y, np.exp(d))
    for _ in range(80):
        if fc < fd: b, d, fd = d, c, fc; c = b - g * (b - a); fc = nll(z, y, np.exp(c))
        else: a, c, fc = c, d, fd; d = a + g * (b - a); fd = nll(z, y, np.exp(d))
    T = float(np.exp((a + b) / 2))
    return {'T': round(T, 4), 'nll_before': round(nll(z, y, 1.0), 4), 'nll_after': round(nll(z, y, T), 4)}

def brier(p, y):
    oh = np.zeros_like(p); oh[np.arange(len(y)), y] = 1
    return float(np.mean(np.sum((p - oh) ** 2, axis=1)))

def ece(p, y, bins=15):
    conf = p.max(1); pred = p.argmax(1); acc = (pred == y).astype(float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for i in range(bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1]) if i else (conf >= edges[0]) & (conf <= edges[1])
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def threshold_table(p, y, groups, thresholds=None):
    """Coverage, selective accuracy and accepted groups for each threshold (see docs/METHODS.md)."""
    thresholds = np.round(np.arange(0.50, 0.995, 0.01), 2) if thresholds is None else thresholds
    conf = p.max(1); pred = p.argmax(1); groups = np.asarray(groups); rows = []
    for t in thresholds:
        acc_mask = conf >= t; n_acc = int(acc_mask.sum())
        rows.append({'threshold': float(t), 'coverage': n_acc / len(y),
                     'selective_accuracy': float((pred[acc_mask] == y[acc_mask]).mean()) if n_acc else None,
                     'selective_risk': float((pred[acc_mask] != y[acc_mask]).mean()) if n_acc else None,
                     'correct_accepted_yield': float(((pred == y) & acc_mask).mean()),
                     'accepted_groups': int(len(set(groups[acc_mask]))), 'accepted': n_acc})
    return rows

def choose_threshold(rows, min_acc=0.95, min_cov=0.50, min_groups=50, fallback=0.90):
    """Highest-coverage threshold with selective accuracy >= 0.95, coverage >= 0.50 and >= 50 accepted groups;
    otherwise 'target infeasible' and the predeclared 0.90 for description only."""
    ok = [r for r in rows if r['selective_accuracy'] is not None and r['selective_accuracy'] >= min_acc
          and r['coverage'] >= min_cov and r['accepted_groups'] >= min_groups]
    if ok:
        best = max(ok, key=lambda r: (r['coverage'], -r['threshold']))
        return {'threshold': best['threshold'], 'status': 'target met', **{k: best[k] for k in ('coverage', 'selective_accuracy', 'accepted_groups')}}
    fb = min(rows, key=lambda r: abs(r['threshold'] - fallback))
    return {'threshold': fallback, 'status': 'target infeasible (0.90 used for description only)',
            **{k: fb[k] for k in ('coverage', 'selective_accuracy', 'accepted_groups')}}

def per_class_coverage(p, y, t, classes):
    conf = p.max(1); out = {}
    for i, c in enumerate(classes):
        m = y == i
        out[c] = round(float((conf[m] >= t).mean()), 4) if m.any() else None
    return out

def select_architecture(agg_rows, size_bytes=None, margin=0.01):
    """Pre-declared selection rule: highest mean selection macro-F1; within `margin` of the best, the smaller converted file."""
    best = max(agg_rows, key=lambda r: r['mean_macro_f1'])
    close = [r for r in agg_rows if best['mean_macro_f1'] - r['mean_macro_f1'] <= margin + 1e-12]
    if len(close) > 1 and size_bytes:
        pick = min(close, key=lambda r: size_bytes[r['architecture']])
        return pick['architecture'], f'within {margin} of best; smallest FP32 file chosen'
    return best['architecture'], (f"highest mean selection macro-F1 ({best['mean_macro_f1']:.4f}); "
                                   f"no other architecture within {margin}")
# ---- OCDF F6 evaluation helpers (Notebook 05): metrics, abstention, group bootstrap ----
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

def full_metrics(p, y, eval_labels, n_out, threshold=None, groups=None):
    """Evaluation metrics (docs/METHODS.md). `eval_labels`: class indices present in this evaluation set (others are 'not evaluated').
    Predictions keep all n_out outputs (never masked), so predicting an absent class counts as an error."""
    pred = p.argmax(1); L = list(eval_labels)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=L, zero_division=0)
    out = {'n_images': int(len(y)), 'n_groups': int(len(set(groups))) if groups is not None else None,
           'accuracy': float(accuracy_score(y, pred)),
           'balanced_accuracy': float(np.mean([np.mean(pred[y == c] == c) for c in L])),
           'macro_f1': float(f1_score(y, pred, labels=L, average='macro', zero_division=0)),
           'per_class': {int(c): {'precision': float(a), 'recall': float(b), 'f1': float(d), 'support': int(s)}
                         for c, a, b, d, s in zip(L, pr, rc, f1, sup)},
           'confusion': confusion_matrix(y, pred, labels=list(range(n_out))).tolist(),
           'nll': float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)))),
           'brier': float(np.mean(np.sum((p - np.eye(n_out)[y]) ** 2, axis=1))),
           'ece_15': ece15(p, y)}
    if threshold is not None:
        acc = p.max(1) >= threshold; n = int(acc.sum())
        out.update({'threshold': float(threshold), 'coverage': float(acc.mean()),
                    'selective_accuracy': float((pred[acc] == y[acc]).mean()) if n else None,
                    'selective_risk': float((pred[acc] != y[acc]).mean()) if n else None,
                    'correct_accepted_yield': float(((pred == y) & acc).mean()),
                    'coverage_per_class': {int(c): float(acc[y == c].mean()) for c in L}})
    return out

def ece15(p, y, bins=15):
    conf = p.max(1); correct = (p.argmax(1) == y).astype(float); e = 0.0
    idx = np.minimum((conf * bins).astype(int), bins - 1)
    for b in range(bins):
        m = idx == b
        if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def fast_accuracy(y, pred):
    return float(np.mean(y == pred))

def fast_macro_f1(y, pred, labels, n_out):
    """Macro-F1 over `labels` from a bincount confusion matrix; identical to sklearn f1_score(average='macro', zero_division=0)."""
    cm = np.bincount(y * n_out + pred, minlength=n_out * n_out).reshape(n_out, n_out)
    tp = np.diag(cm)[labels].astype(float); fp = cm[:, labels].sum(0) - tp; fn = cm[labels, :].sum(1) - tp
    denom = 2 * tp + fp + fn
    return float(np.mean(np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom > 0)))

def group_bootstrap(stat_fn, groups, n_boot=2000, seed=2026):
    """95% percentile interval of stat_fn(index_array), resampling whole groups with replacement (Efron & Tibshirani, 1994)."""
    groups = np.asarray(groups); uniq, inv = np.unique(groups, return_inverse=True)
    order = np.argsort(inv, kind='stable'); lengths = np.bincount(inv); starts = np.concatenate([[0], np.cumsum(lengths)[:-1]])
    rng = np.random.default_rng(seed); vals = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, len(uniq), len(uniq)); lens = lengths[pick]
        offs = np.repeat(starts[pick] - np.cumsum(lens) + lens, lens) + np.arange(lens.sum())
        vals[b] = stat_fn(order[offs])
    return {'estimate': float(stat_fn(np.arange(len(groups)))), 'ci95': [float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))],
            'n_boot': n_boot, 'n_groups': int(len(uniq))}

def reliability_bins(p, y, bins=15):
    conf = p.max(1); correct = (p.argmax(1) == y).astype(float)
    idx = np.minimum((conf * bins).astype(int), bins - 1); rows = []
    for b in range(bins):
        m = idx == b
        rows.append({'bin_low': b / bins, 'bin_high': (b + 1) / bins, 'n': int(m.sum()),
                     'mean_confidence': float(conf[m].mean()) if m.any() else None, 'accuracy': float(correct[m].mean()) if m.any() else None})
    return rows

def risk_coverage(p, y):
    order = np.argsort(-p.max(1)); correct = (p.argmax(1) == y)[order]
    k = np.arange(1, len(y) + 1)
    return k / len(y), 1 - np.cumsum(correct) / k


# 3. Conversion helpers (as Notebook 04), representative sets (the Phase 1 ones), interpreter
def logits_model(path):
    m = keras.models.load_model(path); dense = m.get_layer('probs')
    lin = keras.layers.Dense(dense.units, activation=None, name='logits')
    lm = keras.Model(m.inputs, lin(m.get_layer('dropout').output), name=m.name + '_logits'); lin.set_weights(dense.get_weights()); return m, lm
def make_interpreter(b, threads, xnnpack=True):
    if xnnpack: it = Interpreter(model_content=b, num_threads=threads); path = 'default (XNNPACK)'
    else: it = Interpreter(model_content=b, num_threads=threads, experimental_op_resolver_type=OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES); path = 'builtin kernels without XNNPACK'
    it.allocate_tensors(); return it, path
def _run(b, X, threads, batch, xnnpack):
    it, path = make_interpreter(b, threads, xnnpack); i, o = it.get_input_details()[0]['index'], it.get_output_details()[0]['index']
    it.resize_tensor_input(i, [batch, IMG, IMG, 3]); it.allocate_tensors(); out = []
    for s in range(0, len(X), batch):
        xb = X[s:s + batch].astype(np.float32); n = len(xb)
        if n < batch: xb = np.concatenate([xb, np.zeros((batch - n, IMG, IMG, 3), np.float32)])
        it.set_tensor(i, xb); it.invoke(); out.append(it.get_tensor(o)[:n].copy())
    return np.concatenate(out), path
def run_tflite(b, X, threads=None, batch=32):
    threads = threads or os.cpu_count()
    try: return _run(b, X, threads, batch, True)
    except Exception as e:
        print('  XNNPACK failed (', type(e).__name__, '); using builtin kernels'); return _run(b, X, threads, batch, False)
def inspect(b):
    try: it, path = make_interpreter(b, 1, True)
    except Exception: it, path = make_interpreter(b, 1, False)
    tens = it.get_tensor_details(); dt = pd.Series([t['dtype'].__name__ for t in tens]).value_counts().to_dict()
    try: ops = pd.Series([o['op_name'] for o in it._get_ops_details()]).value_counts().to_dict()
    except Exception as e: ops = {'unavailable': str(e)[:80]}
    return {'tensor_dtypes': dt, 'operators': ops, 'interpreter_path': path}
REP = {}
for crop in CLASSES:
    rp = pd.read_csv(f'{DRIVE}/records/nb04/representative_{crop}.csv', dtype=str)
    rows = U[U.sample_id.isin(set(rp.sample_id))].sort_values('sample_id'); assert len(rows) == len(rp), 'representative set changed'
    REP[crop] = load_array(rows); print(crop, 'representative set (Phase 1):', REP[crop].shape)
def rep_gen(crop): return lambda: ([REP[crop][j:j + 1].astype(np.float32)] for j in range(len(REP[crop])))
def int8_converter(exp_dir, crop):
    c = tf.lite.TFLiteConverter.from_saved_model(exp_dir)
    c.optimizations = [tf.lite.Optimize.DEFAULT]; c.representative_dataset = rep_gen(crop)
    c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]; c.inference_input_type = tf.float32; c.inference_output_type = tf.float32
    return c
def convert_plain(exp_dir, fmt):
    c = tf.lite.TFLiteConverter.from_saved_model(exp_dir)
    if fmt == 'fp16': c.optimizations = [tf.lite.Optimize.DEFAULT]; c.target_spec.supported_types = [tf.float16]
    return c.convert()

In [ ]:
# 4. Quantisation debugger (per-layer error), selective INT8 files for k in {0,5,10,20,40}, FP32/FP16 files, MobileNetV2 comparator (resumable)
try:
    QD = tf.lite.experimental.QuantizationDebugger; QDO = tf.lite.experimental.QuantizationDebugOptions
except AttributeError:
    from tensorflow.lite.tools.optimize.debugging.python.debugger import QuantizationDebugger as QD, QuantizationDebugOptions as QDO   # older/newer layouts
FILES = []                                       # one record per produced file
def file_record(crop, tag, fmt, k, b, path, extra=None):
    r = {'crop': crop, 'model': tag, 'format': fmt, 'k_float_layers': k, 'file': os.path.relpath(path, DRIVE), 'bytes': len(b), 'MiB': round(len(b) / 1048576, 3),
         'sha256': hashlib.sha256(b).hexdigest(), 'inspection': inspect(b)}
    if extra: r.update(extra)
    return r
for crop in CLASSES:
    mdir = f'{MODELS}/{crop}'; os.makedirs(mdir, exist_ok=True)
    for tag in ['phase1_EfficientNetB0', 'shift_EfficientNetB0', 'shift_MobileNetV2']:
        rec_path = f'{OUT}/files_{crop}_{tag}.json'
        if os.path.exists(rec_path): FILES += json.load(open(rec_path)); print('converted already:', crop, tag); continue
        keras.backend.clear_session(); m, lm = logits_model(CAND[(crop, tag)])
        exp_dir = f'/content/export_{crop}_{tag}'; shutil.rmtree(exp_dir, ignore_errors=True); lm.export(exp_dir, verbose=False)
        recs = []; t0 = time.time()
        for fmt in ['fp32', 'fp16']:
            b = convert_plain(exp_dir, fmt); path = f'{mdir}/{crop}_{tag}_{fmt}.tflite'; open(path, 'wb').write(b); recs.append(file_record(crop, tag, fmt, None, b, path))
        if tag == 'shift_MobileNetV2':
            b = int8_converter(exp_dir, crop).convert(); path = f'{mdir}/{crop}_{tag}_int8.tflite'; open(path, 'wb').write(b)
            recs.append(file_record(crop, tag, 'int8', 0, b, path, {'role': 'comparator: full-integer MobileNetV2'}))
        else:
            # (a) debugger run: per-operation quantisation error on the representative images
            dbg = QD(converter=int8_converter(exp_dir, crop), debug_dataset=rep_gen(crop)); dbg.run()
            buf = io.StringIO(); dbg.layer_statistics_dump(buf); buf.seek(0); LS = pd.read_csv(buf)
            LS['rmse_over_scale'] = np.sqrt(LS['mean_squared_error']) / LS['scale'].replace(0, np.nan)
            LS = LS.sort_values('rmse_over_scale', ascending=False).reset_index(drop=True); LS['rank'] = np.arange(1, len(LS) + 1)
            LS.to_csv(f'{OUT}/layer_errors_{crop}_{tag}.csv', index=False)
            print(f'{crop} {tag}: {len(LS)} quantised operations; worst 8 by rmse/scale:')
            print(LS[['rank', 'op_name', 'tensor_name', 'scale', 'rmse_over_scale']].head(8).round(4).to_string(index=False))
            ranked = list(LS.tensor_name)
            # (b) selective INT8: the k worst operations kept in float
            for k in KS:
                deny = ranked[:k]
                dbg_k = QD(converter=int8_converter(exp_dir, crop), debug_dataset=rep_gen(crop), debug_options=QDO(denylisted_nodes=deny)) if k else None
                b = dbg_k.get_nondebug_quantized_model() if k else int8_converter(exp_dir, crop).convert()
                path = f'{mdir}/{crop}_{tag}_int8_k{k}.tflite'; open(path, 'wb').write(b)
                recs.append(file_record(crop, tag, 'int8_selective', k, b, path, {'float_layers': deny}))
                print(f'  k={k}: {recs[-1]["MiB"]} MiB')
        json.dump(recs, open(rec_path, 'w'), indent=1, default=str); FILES += recs
        print(f'{crop} {tag}: {len(recs)} files in {(time.time() - t0) / 60:.1f} min'); shutil.rmtree(exp_dir, ignore_errors=True); del m, lm; gc.collect()
FD = pd.DataFrame([{k: v for k, v in r.items() if k not in ('inspection', 'float_layers')} for r in FILES]); print(FD[['crop', 'model', 'format', 'k_float_layers', 'MiB']].to_string(index=False))

In [ ]:
# 5. Score every file on select / calib_T / thresh (logits saved), T and threshold by the Phase 1 rules, budgets, size-accuracy curve, decision rule 2
RES = []
for crop in CLASSES:
    classes = CLASSES[crop]; X = {}; ys = {p: np.array([classes.index(l) for l in d.research_label]) for p, d in PARTS[crop].items()}
    for r in [f for f in FILES if f['crop'] == crop]:
        stem = os.path.splitext(os.path.basename(r['file']))[0]
        if not all(os.path.exists(f'{LOGITS}/{stem}_{p}.npy') for p in PARTS[crop]):
            b = open(f'{DRIVE}/{r["file"]}', 'rb').read()
            for p in PARTS[crop]:
                if p not in X: X[p] = load_array(PARTS[crop][p])
                t0 = time.time(); z, _ = run_tflite(b, X[p]); np.save(f'{LOGITS}/{stem}_{p}.npy', z); print(f'{stem} {p}: {len(z)} images in {time.time() - t0:.0f}s')
        z = {p: np.load(f'{LOGITS}/{stem}_{p}.npy') for p in PARTS[crop]}
        ps = softmax(z['select']); cal = fit_temperature(z['calib_T'], ys['calib_T']); pt = softmax(z['thresh'], cal['T'])
        curve = threshold_table(pt, ys['thresh'], PARTS[crop]['thresh'].group_id.values); ch = choose_threshold(curve)
        RES.append({**{k: r[k] for k in ['crop', 'model', 'format', 'k_float_layers', 'file', 'bytes', 'MiB', 'sha256']},
                    'select_accuracy': float(accuracy_score(ys['select'], ps.argmax(1))), 'select_macro_f1': float(f1_score(ys['select'], ps.argmax(1), average='macro')),
                    'T': cal['T'], 'calib_nll_after': cal['nll_after'], 'calib_ece_after': ece(softmax(z['calib_T'], cal['T']), ys['calib_T']),
                    'threshold': ch['threshold'], 'threshold_status': ch['status'], 'thresh_coverage': ch['coverage'], 'thresh_selective_accuracy': ch['selective_accuracy']})
    del X; gc.collect()
R = pd.DataFrame(RES)
fp = R[R.format == 'fp32'].set_index(['crop', 'model'])
R['acc_change_vs_fp32_pp'] = [100 * (r.select_accuracy - fp.loc[(r.crop, r.model), 'select_accuracy']) for r in R.itertuples()]
R['macro_f1_change_vs_fp32'] = [r.select_macro_f1 - fp.loc[(r.crop, r.model), 'select_macro_f1'] for r in R.itertuples()]
R['size_ok'] = R.MiB <= 10; R['accuracy_ok'] = R.acc_change_vs_fp32_pp >= -2; R['macro_f1_ok'] = R.macro_f1_change_vs_fp32 >= -0.02
R['passes_budgets'] = R.size_ok & R.accuracy_ok & R.macro_f1_ok
R.to_csv(f'{OUT}/e14_files_scored.csv', index=False)
pd.set_option('display.width', 250)
print(R[['crop', 'model', 'format', 'k_float_layers', 'MiB', 'select_accuracy', 'select_macro_f1', 'acc_change_vs_fp32_pp', 'macro_f1_change_vs_fp32', 'passes_budgets', 'T', 'threshold', 'threshold_status']].round(4).to_string(index=False))
# Decision rule 2: for the v3 model of each crop, the smallest file passing every budget among selective INT8 (smallest passing k) and FP16
RULE2 = {}
for crop in CLASSES:
    tag = RULE1[crop]['v3_model']; d = R[(R.crop == crop) & (R.model == tag)]
    sel_pass = d[(d.format == 'int8_selective') & d.passes_budgets].sort_values('k_float_layers')
    cands = []
    if len(sel_pass): cands.append(sel_pass.iloc[0])
    f16 = d[d.format == 'fp16']
    if len(f16) and bool(f16.iloc[0].passes_budgets): cands.append(f16.iloc[0])
    if not cands: cands.append(d[d.format == 'fp32'].iloc[0])
    best = min(cands, key=lambda r: r.bytes)
    RULE2[crop] = {'v3_model': tag, 'candidates': [{'format': c.format, 'k': None if pd.isna(c.k_float_layers) else int(c.k_float_layers), 'MiB': float(c.MiB)} for c in cands],
                   'chosen': {'format': best.format, 'k_float_layers': None if pd.isna(best.k_float_layers) else int(best.k_float_layers), 'file': best.file, 'MiB': float(best.MiB), 'sha256': best.sha256},
                   'first_k_passing_budgets': None if not len(sel_pass) else int(sel_pass.iloc[0].k_float_layers),
                   'full_int8_k0': d[(d.format == 'int8_selective') & (d.k_float_layers == 0)][['MiB', 'acc_change_vs_fp32_pp', 'macro_f1_change_vs_fp32', 'passes_budgets']].iloc[0].to_dict(),
                   'mobilenetv2_full_int8_comparator': R[(R.crop == crop) & (R.model == 'shift_MobileNetV2') & (R.format == 'int8')][['MiB', 'select_macro_f1', 'acc_change_vs_fp32_pp', 'passes_budgets']].iloc[0].to_dict()}
    print(crop, 'rule 2:', json.dumps(RULE2[crop]['chosen']), '| first passing k:', RULE2[crop]['first_k_passing_budgets'])
both = sum(RULE2[c]['chosen']['MiB'] for c in CLASSES); print('both crops:', round(both, 3), 'MiB (budget 20)')
# Figure: size and macro-F1 against k
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, crop in zip(axes, CLASSES):
    for tag, mk in [('phase1_EfficientNetB0', 'o'), ('shift_EfficientNetB0', 's')]:
        d = R[(R.crop == crop) & (R.model == tag) & (R.format == 'int8_selective')].sort_values('k_float_layers')
        ax.plot(d.k_float_layers, d.select_macro_f1, marker=mk, label=f'{tag} selective INT8')
        ax.axhline(fp.loc[(crop, tag), 'select_macro_f1'], linestyle=':', color='grey')
        for r in d.itertuples(): ax.annotate(f'{r.MiB:.1f} MiB', (r.k_float_layers, r.select_macro_f1), textcoords='offset points', xytext=(0, 6), fontsize=7, ha='center')
    ax.set_xlabel('k operations kept in float'); ax.set_ylabel('selection macro-F1'); ax.set_title(f'{crop}: selective INT8 (dotted = FP32)'); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(f'{OUT}/figure_e14_selective_int8.png', dpi=130); plt.show(); plt.close(fig)

In [ ]:
# 6. Abstention rules for the v3 instance: C1 per-class thresholds (threshold partition), C2 energy score with the 95th-percentile rule
def energy(z, T): return -T * np.log(np.exp(z / T - (z / T).max(1, keepdims=True)).sum(1)) - (z / T).max(1) * T
C1C2 = {}
for crop in CLASSES:
    classes = CLASSES[crop]; ch = RULE2[crop]['chosen']; stem = os.path.splitext(os.path.basename(ch['file']))[0]
    z = {p: np.load(f'{LOGITS}/{stem}_{p}.npy') for p in PARTS[crop]}; ys = {p: np.array([classes.index(l) for l in d.research_label]) for p, d in PARTS[crop].items()}
    row = R[R.file == ch['file']].iloc[0]; T = float(row['T']); pt = softmax(z['thresh'], T); pred = pt.argmax(1); groups = PARTS[crop]['thresh'].group_id.values
    per_class = {}
    for i, c in enumerate(classes):          # C1: among images predicted as class c, the highest-coverage threshold with selective precision >= 0.95 (Phase 1 rule applied per class)
        m = pred == i
        if m.sum() == 0: per_class[c] = {'status': 'no predictions of this class on the threshold partition'}; continue
        curve = threshold_table(pt[m], ys['thresh'][m], groups[m]); chc = choose_threshold(curve, min_groups=min(50, max(5, int(len(set(groups[m])) * 0.5))))
        per_class[c] = {'n_predicted': int(m.sum()), **chc}
    all_met = all(v.get('status') == 'target met' for v in per_class.values())
    e_thr = energy(z['thresh'], T); e95 = float(np.percentile(e_thr, 95))     # C2: reject when energy exceeds the 95th percentile of in-distribution energies
    C1C2[crop] = {'file': ch['file'], 'T': T, 'global_threshold': float(row.threshold), 'global_status': row.threshold_status,
                  'C1_per_class': per_class, 'C1_adopted': bool(all_met),
                  'v3_abstention': ('C1 per-class thresholds' if all_met else 'Phase 1 global rule (C1 target not met for every class)'),
                  'C2_energy': {'rule': 'score only: reject if energy > 95th percentile of threshold-partition energies', 'energy_p95': e95,
                                'energy_mean_thresh': float(e_thr.mean()), 'energy_sd_thresh': float(e_thr.std())}}
    print(f'\n{crop}: C1 per class:'); print(pd.DataFrame(per_class).T.to_string()); print('C1 adopted:', all_met, '| energy p95:', round(e95, 4))
json.dump(C1C2, open(f'{OUT}/e14_abstention_rules.json', 'w'), indent=1, default=float)

In [ ]:
# 7. FREEZE v3 (unseals d5_test for Notebook 11), results zip
v3 = {}
for crop in CLASSES:
    ch = RULE2[crop]['chosen']; row = R[R.file == ch['file']].iloc[0]; tag = RULE1[crop]['v3_model']
    v3[crop] = {'model': tag, 'keras_weights': os.path.relpath(CAND[(crop, tag)], DRIVE), 'architecture': 'EfficientNetB0', 'seed': 42,
                'format': ch['format'], 'k_float_layers': ch['k_float_layers'], 'file': ch['file'], 'sha256': ch['sha256'], 'bytes': int(row.bytes), 'MiB': float(row.MiB),
                'labels_in_output_order': CLASSES[crop], 'temperature_T': float(row['T']), 'threshold': float(row.threshold), 'threshold_status': row.threshold_status,
                'abstention': C1C2[crop]['v3_abstention'], 'C1_per_class': C1C2[crop]['C1_per_class'], 'C2_energy_p95': C1C2[crop]['C2_energy']['energy_p95'],
                'input': 'float32 [1,224,224,3], RGB, values 0-255, aspect-preserving resize with zero padding (bilinear)',
                'output': 'logits; probabilities = softmax(logits / T); abstain if max probability < threshold (global) or < the predicted class threshold (C1)'}
frozen = {'protocol_version': 'v3', 'registration_sha256': reg_sha, 'manifest_v3_sha256': seal['sha256'], 'frozen_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
          'rule1_model': RULE1, 'rule2_format': RULE2, 'rule3_adaptation': ADEC['maize'], 'adaptation_registration_sha256': open(adec_path + '.sha256').read().strip(),
          'rule4_abstention': {c: C1C2[c]['v3_abstention'] for c in CLASSES}, 'deployed_instance_v3': v3, 'phase1_deployed_instance': DEP,
          'all_files': R.astype(object).where(pd.notna(R), None).to_dict('records'), 'both_crops_MiB': float(sum(v3[c]['MiB'] for c in CLASSES)),
          'note': 'Frozen before any test half, challenge set or D5 image was loaded in Phase 2 scoring. d5_test is unsealed by this file.', 'environment': env}
f3 = f'{DRIVE}/records/frozen_choices_v3.json'
assert not os.path.exists(f3); json.dump(frozen, open(f3, 'w'), indent=1, default=float)
sha3 = hashlib.sha256(open(f3, 'rb').read()).hexdigest(); open(f'{DRIVE}/records/frozen_choices_v3.sha256', 'w').write(sha3)
print('FROZEN v3:', f3, sha3)
for c in CLASSES: print(c, '->', v3[c]['model'], v3[c]['format'], 'k=', v3[c]['k_float_layers'], f'{v3[c]["MiB"]} MiB', 'T', v3[c]['temperature_T'], 'thr', v3[c]['threshold'], '|', v3[c]['abstention'])
import zipfile
zp = '/content/nb10_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f'{OUT}/*'):
        if os.path.isfile(p): z.write(p, os.path.basename(p))
    for p in [f3, f3.replace('.json', '.sha256')]: z.write(p, os.path.basename(p))
from google.colab import files
files.download(zp); print('nb10_results.zip ready')

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb10_results.zip`. The copies used for the published results are in this repository under `records/nb10/`.